## Example 1: Converse against a local stand-in for Bedrock

In [ ]:
import importlib.metadata

from aiohttp import web
from aws_sdk_bedrock_runtime.client import AsyncBedrockRuntimeClient
from aws_sdk_bedrock_runtime.config import AsyncBedrockRuntimeConfig
from aws_sdk_bedrock_runtime.models import ContentBlockText, ConverseInput, Message

# Example 1: the real client, signer and JSON protocol, pointed at a local HTTP server that answers like Bedrock
assert importlib.metadata.version("aws-sdk-bedrock-runtime") == "0.12.0"
seen = {}
async def converse(request):
    seen.update(path=request.path, auth=request.headers.get("Authorization", ""), body=await request.json())
    return web.json_response({"output": {"message": {"role": "assistant", "content": [{"text": "4"}]}}, "stopReason": "end_turn",
                              "usage": {"inputTokens": 7, "outputTokens": 1, "totalTokens": 8}, "metrics": {"latencyMs": 3}})
app = web.Application(); app.router.add_post("/model/{model}/converse", converse)
runner = web.AppRunner(app); await runner.setup(); site = web.TCPSite(runner, "127.0.0.1", 0); await site.start()
port = site._server.sockets[0].getsockname()[1]
client = AsyncBedrockRuntimeClient(await AsyncBedrockRuntimeConfig.resolve(endpoint_uri=f"http://127.0.0.1:{port}", region="us-east-1", aws_access_key_id="AK", aws_secret_access_key="SK"))
out = await client.converse(ConverseInput(model_id="amazon.nova-micro-v1:0", messages=[Message(role="user", content=[ContentBlockText(value="2+2?")])]))
await runner.cleanup()
assert out.output.value.content[0].value == "4" and out.stop_reason == "end_turn" and out.usage.total_tokens == 8, out
assert seen["path"] == "/model/amazon.nova-micro-v1:0/converse" and seen["auth"].startswith("AWS4-HMAC-SHA256") and "/bedrock/aws4_request" in seen["auth"]
assert seen["body"]["messages"] == [{"role": "user", "content": [{"text": "2+2?"}]}], seen["body"]
print("Example 1 OK:", out.output.value.content[0].value, out.usage)

## Example 2: Request models serialize to Bedrock's JSON

In [ ]:
from aws_sdk_bedrock_runtime.models import ContentBlockText, ConverseInput, InferenceConfiguration, Message, SystemContentBlockText
from smithy_json import JSONCodec

# Example 2: the generated dataclasses carry their Smithy schema; the JSON codec writes the wire form
request = ConverseInput(model_id="m", messages=[Message(role="user", content=[ContentBlockText(value="hi")])],
                        system=[SystemContentBlockText(value="Be brief.")], inference_config=InferenceConfiguration(max_tokens=64, temperature=0.2))
raw = JSONCodec().serialize(request)
assert b'"system":[{"text":"Be brief."}]' in raw and b'"maxTokens":64' in raw and b'"temperature":0.2' in raw, raw
print("Example 2 OK:", raw)

## Example 3: Configuration resolves from the environment

In [ ]:
import os

from aws_sdk_bedrock_runtime.config import AsyncBedrockRuntimeConfig

# Example 3: resolve() takes region and retry settings from AWS_* variables when they are not passed
os.environ.update(AWS_REGION="eu-central-1", AWS_MAX_ATTEMPTS="5", AWS_RETRY_MODE="standard")
config = await AsyncBedrockRuntimeConfig.resolve()
assert config.region == "eu-central-1" and config.max_attempts == 5 and config.retry_mode == "standard", (config.region, config.max_attempts, config.retry_mode)
assert (await AsyncBedrockRuntimeConfig.resolve(region="us-west-2")).region == "us-west-2"
print("Example 3 OK:", config.region, config.max_attempts)

## Example 4: Service errors come back as typed exceptions

In [ ]:
from aiohttp import web
from aws_sdk_bedrock_runtime.client import AsyncBedrockRuntimeClient
from aws_sdk_bedrock_runtime.config import AsyncBedrockRuntimeConfig
from aws_sdk_bedrock_runtime.models import ContentBlockText, ConverseInput, Message, ValidationException

# Example 4: a 400 with Bedrock's error header is raised as the matching modeled exception
async def reject(request):
    return web.json_response({"message": "model id is not valid"}, status=400, headers={"x-amzn-ErrorType": "ValidationException"})
app = web.Application(); app.router.add_post("/model/{model}/converse", reject)
runner = web.AppRunner(app); await runner.setup(); site = web.TCPSite(runner, "127.0.0.1", 0); await site.start()
port = site._server.sockets[0].getsockname()[1]
client = AsyncBedrockRuntimeClient(await AsyncBedrockRuntimeConfig.resolve(endpoint_uri=f"http://127.0.0.1:{port}", region="us-east-1", aws_access_key_id="AK", aws_secret_access_key="SK", max_attempts=1))
try:
    await client.converse(ConverseInput(model_id="nope", messages=[Message(role="user", content=[ContentBlockText(value="x")])]))
    raise AssertionError("the 400 was not raised")
except ValidationException as error:
    print("Example 4 OK:", type(error).__name__, error)
finally:
    await runner.cleanup()

## Example 5: Tool definitions for function calling

In [ ]:
from aws_sdk_bedrock_runtime.models import ToolConfiguration, ToolInputSchemaJson, ToolSpecification, ToolToolSpec
from smithy_core.documents import Document
from smithy_json import JSONCodec

# Example 5: a tool the model may call, with a JSON schema for its input
schema = Document({"type": Document("object"), "properties": Document({"city": Document({"type": Document("string")})})})
config = ToolConfiguration(tools=[ToolToolSpec(value=ToolSpecification(name="weather", description="Weather by city", input_schema=ToolInputSchemaJson(value=schema)))])
raw = JSONCodec().serialize(config.tools[0])
assert b'"name":"weather"' in raw and b'"city"' in raw, raw
print("Example 5 OK:", raw[:120])